# Neuronal Tracer — Part 1 / Step 1: Load → Preprocess → Tubularity

**Output:** `output/tubularity.npz`, `output/stack_iso.tif`, `output/T_combined.tif`  
**Next:** Run `neuronal_tracer1_step2.ipynb` for skeletonization.

| Step | Description |
|------|-------------|
| 1 | Data loading + metadata |
| 2 | Visualization utilities |
| 3 | Preprocessing (downsample → normalize → isotropic rescale) |
| 4 | Multi-scale tubularity (Frangi + Meijering) |

In [ ]:
# Run once to install dependencies, then restart kernel
# !pip install tifffile scikit-image scipy numpy matplotlib ipywidgets plotly networkx


In [ ]:
import numpy as np
import tifffile
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from scipy import ndimage
from skimage import filters, morphology, measure
from skimage.filters import frangi, meijering
import plotly.graph_objects as go
import networkx as nx
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 100,
    'figure.facecolor': '#111111',
    'axes.facecolor': '#111111',
    'axes.titlecolor': 'white',
    'text.color': 'white'
})
print('All packages loaded.')


---
## Configuration
Set all parameters here. Re-run this cell after any change.


In [ ]:
# ── File ──────────────────────────────────────────────────────────────────────
DATA_PATH = 'data/sample_crop.tif'

# ── Voxel size ────────────────────────────────────────────────────────────────
# XY is auto-detected from TIFF metadata (should be ~0.171 µm for this file).
# Z: enter your microscope z-step in µm (check acquisition settings).
VOXEL_Z = 1.0      # <-- UPDATE with your actual z-step in µm

# ── Working resolution ────────────────────────────────────────────────────────
# Downsample for faster development. Set to 1 for final production run.
DOWNSAMPLE_XY = 2  # 2 = halve X and Y resolution
DOWNSAMPLE_Z  = 1  # usually keep at 1

# ── Multi-scale tubularity ────────────────────────────────────────────────────
SIGMA_MIN = 0.5    # min tube radius in voxels (after isotropic rescaling)
SIGMA_MAX = 5.0    # max tube radius in voxels
N_SIGMAS  = 8      # number of log-spaced scales between SIGMA_MIN and SIGMA_MAX

# ── Candidate extraction ──────────────────────────────────────────────────────
TUBULARITY_THRESHOLD = 0.10   # fraction of max response [0-1]; lower = more candidates
MIN_COMPONENT_VOXELS = 50     # discard connected components smaller than this

print('Configuration loaded.')


In [ ]:
# ── Load tubularity checkpoint (skip Steps 3-4 if already computed) ───────────
# Uncomment to restore from a previous run:

# ck = np.load('output/tubularity.npz')
# T_combined  = ck['T_combined']
# scale_idx_m = ck['scale_idx_m']
# radius_map  = ck['radius_map']
# sigmas      = ck['sigmas']
# voxel_iso   = float(ck['voxel_iso'])
# stack_iso   = tifffile.imread('output/stack_iso.tif')
# print(f'Tubularity checkpoint loaded — T_combined={T_combined.shape}')

---
## Step 1 · Data Loading & Inspection


In [ ]:
print(f'Loading: {DATA_PATH}')
stack_raw = tifffile.imread(DATA_PATH)  # axis order: (Z, Y, X)

if stack_raw.ndim == 2:
    stack_raw = stack_raw[np.newaxis]

print(f'  Shape (Z, Y, X)  : {stack_raw.shape}')
print(f'  dtype            : {stack_raw.dtype}')
print(f'  Memory           : {stack_raw.nbytes / 1e9:.2f} GB')
print(f'  Intensity range  : {int(stack_raw.min())} – {int(stack_raw.max())}')


In [ ]:
# Auto-detect XY voxel size from TIFF metadata
voxel_xy = None

with tifffile.TiffFile(DATA_PATH) as tif:
    if tif.is_imagej:
        ij = tif.imagej_metadata
        print('ImageJ metadata:')
        for k, v in ij.items():
            if k != 'labels':
                print(f'  {k}: {v}')
        # Z spacing may be stored under 'spacing'
        if 'spacing' in ij:
            print(f"\n  Z spacing in file: {ij['spacing']} {ij.get('unit','')}")
            print(f"  -> You can set VOXEL_Z = {ij['spacing']} in Configuration")

    xres = tif.pages[0].tags.get('XResolution')
    if xres is not None:
        val = xres.value
        res = (val[0] / val[1]) if isinstance(val, tuple) and val[1] != 0 else float(val)
        if res > 0:
            voxel_xy = 1.0 / res

if voxel_xy is None or voxel_xy <= 0:
    voxel_xy = 1.0
    print('\nWARNING: XY voxel size not found. Set voxel_xy manually in next cell.')
else:
    print(f'\nDetected XY voxel size : {voxel_xy:.4f} µm/pixel')

print(f'Z voxel size (config)  : {VOXEL_Z:.4f} µm/slice')
print(f'Anisotropy (Z/XY)      : {VOXEL_Z / voxel_xy:.2f}x')


---
## Step 2 · Visualization Utilities
Run once — these functions are reused throughout the notebook.


In [ ]:
def _aniso():
    """Anisotropy of the *downsampled* working stack."""
    return (VOXEL_Z * DOWNSAMPLE_Z) / (voxel_xy * DOWNSAMPLE_XY)


def show_mip(vol, title='Max Intensity Projection', cmap='gray', vmin=None, vmax=None,
             ar=None):
    """Three-axis maximum intensity projections.
    ar: Z/XY aspect ratio for XZ/YZ panels; defaults to _aniso() (downsampled stack).
        Pass VOXEL_Z/voxel_xy for the raw stack, or 1.0 for an isotropic stack."""
    if ar is None:
        ar = _aniso()
    fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
    kw = dict(cmap=cmap, vmin=vmin, vmax=vmax, interpolation='nearest')
    axes[0].imshow(vol.max(axis=0), aspect='equal', **kw)
    axes[0].set_title('XY  (Z projection)')
    axes[1].imshow(vol.max(axis=1), aspect=ar, **kw)
    axes[1].set_title('XZ  (Y projection)')
    axes[2].imshow(vol.max(axis=2), aspect=ar, **kw)
    axes[2].set_title('YZ  (X projection)')
    for ax in axes:
        ax.axis('off')
    fig.suptitle(title, fontsize=13)
    plt.show()


def show_slices(vol, title='', cmap='gray', vmin=None, vmax=None,
                overlay=None, overlay_cmap='hot', overlay_thresh=0.1,
                ar=None):
    """Interactive Z-slice browser with optional colour overlay.
    ar: Z/XY aspect ratio for XZ/YZ panels; defaults to _aniso() (downsampled stack).
        Pass VOXEL_Z/voxel_xy for the raw stack, or 1.0 for an isotropic stack."""
    if ar is None:
        ar = _aniso()
    nz  = vol.shape[0]
    out = widgets.Output()

    slider = widgets.IntSlider(
        value=nz // 2, min=0, max=nz - 1,
        description='Z slice', layout=widgets.Layout(width='600px')
    )
    tslider = widgets.FloatSlider(
        value=overlay_thresh, min=0.01, max=0.99, step=0.01,
        description='Overlay >=', layout=widgets.Layout(width='400px')
    ) if overlay is not None else None

    def _draw(z, thresh=overlay_thresh):
        with out:
            clear_output(wait=True)
            ncols = 4 if overlay is not None else 3
            fig, axes = plt.subplots(1, ncols, figsize=(5 * ncols, 5),
                                     constrained_layout=True)
            kw = dict(cmap=cmap, vmin=vmin, vmax=vmax, interpolation='nearest')

            axes[0].imshow(vol[z], aspect='equal', **kw)
            axes[0].set_title(f'XY  z={z}')

            axes[1].imshow(vol[:, vol.shape[1] // 2, :], aspect=ar, **kw)
            axes[1].axhline(z, color='red', lw=0.8, alpha=0.7)
            axes[1].set_title('XZ  y=center')

            axes[2].imshow(vol[:, :, vol.shape[2] // 2], aspect=ar, **kw)
            axes[2].axhline(z, color='red', lw=0.8, alpha=0.7)
            axes[2].set_title('YZ  x=center')

            if overlay is not None:
                axes[3].imshow(vol[z], aspect='equal', **kw)
                ov = overlay[z].astype(float)
                ov_masked = np.ma.masked_where(ov < thresh * overlay.max(), ov)
                axes[3].imshow(ov_masked, cmap=overlay_cmap, alpha=0.75,
                               aspect='equal', interpolation='nearest')
                axes[3].set_title(f'Overlay  z={z}')

            for ax in axes:
                ax.axis('off')
            fig.suptitle(title, fontsize=12)
            plt.show()

    if tslider is not None:
        widgets.interactive_output(_draw, {'z': slider, 'thresh': tslider})
        display(widgets.VBox([slider, tslider]), out)
    else:
        widgets.interactive_output(_draw, {'z': slider})
        display(slider, out)
    _draw(nz // 2)


def compare_mip(va, vb, title_a='Before', title_b='After', cmap='gray',
                ar_a=None, ar_b=None):
    """Side-by-side MIP comparison (2 rows x 3 axes).
    ar_a/ar_b: Z/XY aspect ratios for each row; both default to _aniso().
        Pass 1.0 for an isotropic stack (e.g. stack_iso after Z-rescaling)."""
    if ar_a is None:
        ar_a = _aniso()
    if ar_b is None:
        ar_b = _aniso()
    fig, axes = plt.subplots(2, 3, figsize=(16, 9), constrained_layout=True)
    for row, (vol, ttl, ar) in enumerate([(va, title_a, ar_a), (vb, title_b, ar_b)]):
        axes[row, 0].imshow(vol.max(axis=0), cmap=cmap, aspect='equal')
        axes[row, 0].set_title(f'{ttl} – XY')
        axes[row, 1].imshow(vol.max(axis=1), cmap=cmap, aspect=ar)
        axes[row, 1].set_title(f'{ttl} – XZ')
        axes[row, 2].imshow(vol.max(axis=2), cmap=cmap, aspect=ar)
        axes[row, 2].set_title(f'{ttl} – YZ')
    for ax in axes.ravel():
        ax.axis('off')
    plt.show()


print('Visualization utilities ready.')


In [ ]:
show_mip(stack_raw, title='Raw Stack', ar=VOXEL_Z / voxel_xy)


In [ ]:
show_slices(stack_raw, title='Raw Stack — Slice Browser', ar=VOXEL_Z / voxel_xy)


---
## Step 3 · Preprocessing

1. **Downsample** XY for faster iteration (controlled by `DOWNSAMPLE_XY`)
2. **Normalize** to float32 [0, 1] with percentile clipping to suppress hot pixels
3. **Rescale Z** to match XY spacing — this makes voxels isotropic, which is required for the Hessian-based tubularity filters to work correctly

> **Memory note:** rescaling Z by the anisotropy factor multiplies Z-slice count by that factor.  
> If RAM is tight, increase `DOWNSAMPLE_XY` instead of skipping this step.


In [ ]:
# 3a. Downsample
stack_ds = stack_raw[::DOWNSAMPLE_Z, ::DOWNSAMPLE_XY, ::DOWNSAMPLE_XY]
vxy = voxel_xy * DOWNSAMPLE_XY
vz  = VOXEL_Z  * DOWNSAMPLE_Z
aniso = vz / vxy

print(f'Shape after downsample : {stack_ds.shape}  (was {stack_raw.shape})')
print(f'XY voxel size          : {vxy:.4f} µm')
print(f'Z  voxel size          : {vz:.4f} µm')
print(f'Anisotropy (Z/XY)      : {aniso:.2f}x')
print(f'Memory                 : {stack_ds.nbytes / 1e9:.2f} GB')


In [ ]:
# 3b. Normalize to float32 [0, 1]
stack_f = stack_ds.astype(np.float32)
p_low, p_high = np.percentile(stack_f, [1.0, 99.5])

print(f'Full range     : {stack_f.min():.0f} – {stack_f.max():.0f}')
print(f'Clipping to    : {p_low:.0f} – {p_high:.0f}  (1st–99.5th percentile)')

stack_norm = np.clip(stack_f, p_low, p_high)
stack_norm = (stack_norm - p_low) / (p_high - p_low)
print(f'Normalized     : {stack_norm.min():.4f} – {stack_norm.max():.4f}')


In [ ]:
# 3c. Isotropic rescaling: stretch Z to match XY voxel size
if aniso > 1.2:
    print(f'Rescaling Z axis by {aniso:.2f}x to achieve isotropic voxels ...')
    stack_iso = ndimage.zoom(stack_norm, (aniso, 1.0, 1.0), order=1, prefilter=False)
    voxel_iso = vxy
    print(f'  Before : {stack_norm.shape}')
    print(f'  After  : {stack_iso.shape}')
    print(f'  Memory : {stack_iso.nbytes / 1e9:.2f} GB')
else:
    stack_iso = stack_norm
    voxel_iso = vxy
    print('Voxels already near-isotropic — no Z rescaling needed.')

print(f'\nIsotropic voxel size: {voxel_iso:.4f} µm')


In [ ]:
compare_mip(stack_norm, stack_iso,
            title_a='Before Z-rescaling', title_b='After Z-rescaling (isotropic)',
            ar_b=1.0)


---
## Step 4 · Multi-scale Tubularity Estimation

For each scale σ the **Hessian matrix** is computed, its eigenvalues λ₁ ≤ λ₂ ≤ λ₃ analysed, and a tubularity score assigned:

| Filter | Core idea | Recommended for |
|--------|-----------|----------------|
| **Frangi** | λ₁ ≈ 0, λ₂ ≈ λ₃ < 0 → tube | Dendrites, thicker processes |
| **Meijering** | Modified Frangi without blob penalty | Thin axons, fine varicosities |

**Key outputs:**
- `T_combined` — tubularity map (geometric mean of Frangi + Meijering)
- `radius_map` — local tube radius in µm, from the scale σ* at which each voxel scores highest


In [ ]:
sigmas = np.geomspace(SIGMA_MIN, SIGMA_MAX, N_SIGMAS)
print(f'Scales σ (voxels)   : {np.round(sigmas, 3)}')
print(f'Tube diameters (µm) : {np.round(sigmas * 2 * voxel_iso, 3)}')
print(f'Stack shape         : {stack_iso.shape}')
print(f'\nEstimated peak memory (per-scale): ~{stack_iso.nbytes / 1e9:.1f} GB')
print('(Scales processed one at a time to avoid stacking all in memory)')


In [ ]:
import gc

# Free preprocessing intermediates that are no longer needed
for _var in ('stack_ds', 'stack_f'):
    if _var in dir():
        del globals()[_var]
gc.collect()

SLAB_SIZE = 64
OVERLAP   = int(np.ceil(SIGMA_MAX * 3))
nz        = stack_iso.shape[0]
n_slabs   = int(np.ceil(nz / SLAB_SIZE))

# Only two full-volume arrays needed (vs four previously).
# T_frangi / T_meijering are never materialised at full volume.
T_combined  = np.zeros(stack_iso.shape, dtype=np.float32)
scale_idx_m = np.zeros(stack_iso.shape, dtype=np.uint8)

max_f_global = 0.0   # tracked as scalars for final normalisation
max_m_global = 0.0

print(f'Persistent arrays  : T_combined {T_combined.nbytes/1e6:.0f} MB'
      f' + scale_idx_m {scale_idx_m.nbytes/1e6:.0f} MB')
print(f'Slab size : {SLAB_SIZE}  |  Overlap : {OVERLAP}  |  Slabs : {n_slabs}\n')

for slab_i, z0 in enumerate(range(0, nz, SLAB_SIZE)):
    z1  = min(z0 + SLAB_SIZE, nz)
    z0p = max(0, z0 - OVERLAP)
    z1p = min(nz, z1 + OVERLAP)
    core = slice(z0 - z0p, z1 - z0p)
    sz   = z1 - z0

    print(f'  slab {slab_i+1}/{n_slabs}  z={z0}-{z1}', end='  sigmas:', flush=True)

    # Per-slab running-max accumulators (float16 — values in [0,1])
    T_f_slab = np.zeros((sz, stack_iso.shape[1], stack_iso.shape[2]), dtype=np.float16)
    T_m_slab = np.zeros_like(T_f_slab)
    slab     = stack_iso[z0p:z1p]          # view, no copy

    for i, sigma in enumerate(sigmas):
        print(f' {i+1}', end='', flush=True)
        sf = frangi(slab, sigmas=[sigma], black_ridges=False,
                    alpha=0.5, beta=0.5).astype(np.float32)
        sm = meijering(slab, sigmas=[sigma], black_ridges=False).astype(np.float32)

        sf_c, sm_c = sf[core], sm[core]

        upd_f = sf_c > T_f_slab
        T_f_slab[upd_f] = sf_c[upd_f]

        upd_m = sm_c > T_m_slab
        T_m_slab[upd_m]         = sm_c[upd_m]
        scale_idx_m[z0:z1][upd_m] = i

        del sf, sm, sf_c, sm_c, upd_f, upd_m

    # Track global maxima (scalars) for normalisation pass
    max_f_global = max(max_f_global, float(T_f_slab.max()))
    max_m_global = max(max_m_global, float(T_m_slab.max()))

    # Write unnormalized geometric-mean score; normalise globally below
    T_combined[z0:z1] = np.sqrt(T_f_slab.astype(np.float32) *
                                 T_m_slab.astype(np.float32))

    del T_f_slab, T_m_slab
    gc.collect()
    print()

# Single in-place normalisation — no extra full-volume copy
T_combined /= (np.sqrt(max_f_global * max_m_global) + 1e-10)

radius_map = sigmas[scale_idx_m] * voxel_iso

print(f'\nCombined  : {T_combined.min():.4f} – {T_combined.max():.4f}')
print(f'Radius    : {radius_map.min():.3f} – {radius_map.max():.3f} µm')


In [ ]:
import os
os.makedirs('output', exist_ok=True)

# ── Save tubularity checkpoint ────────────────────────────────────────────────
np.savez_compressed('output/tubularity.npz',
    T_combined  = T_combined,
    scale_idx_m = scale_idx_m,
    radius_map  = radius_map,
    sigmas      = sigmas,
    voxel_iso   = np.float32(voxel_iso),
)
tifffile.imwrite('output/stack_iso.tif',   stack_iso,  photometric='minisblack')
tifffile.imwrite('output/T_combined.tif',  T_combined, photometric='minisblack')

print('Saved → output/tubularity.npz')
print('Saved → output/stack_iso.tif')
print('Saved → output/T_combined.tif')


In [ ]:
# Radius map: where is each tube wide vs thin?
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)

im0 = axes[0].imshow(T_combined.max(axis=0), cmap='hot', aspect='equal')
axes[0].set_title('Combined tubularity — XY MIP')
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(radius_map.max(axis=0), cmap='viridis', aspect='equal',
                     vmin=SIGMA_MIN * voxel_iso, vmax=SIGMA_MAX * voxel_iso)
axes[1].set_title('Local radius (µm) — XY MIP')
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], label='µm')

plt.show()


In [ ]:
# Interactive slice browser with tubularity overlay
# Use the overlay threshold slider to tune detection sensitivity
show_slices(stack_iso, title='Tubularity Overlay',
            overlay=T_combined, overlay_thresh=0.1)


---
## Roadmap

| Step | Status | Description |
|------|--------|-------------|
| 1 | ✅ | Data loading + metadata |
| 2 | ✅ | Visualization utilities |
| 3 | ✅ | Preprocessing + isotropic rescaling |
| 4 | ✅ | Multi-scale Frangi + Meijering tubularity |
| → | | Continue in **neuronal_tracer1_step2.ipynb** |